# Comparison with Observations: Survey Populations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import pathlib
from scipy.integrate import quad
from scipy import stats

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import utilities.plot_settings

from mlpoppyns.generator.generate_observed_data import load_atnf_meerkat_catalog, load_xray_catalog

In [ ]:
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2

# Auxiliary quantity beta as defined in eq. (72) of Pons & Vigano (2019).
beta = np.pi**2 * cfg["NS_radius"] ** 6 / (NS_inertia * const.C**3)

# Assume an inclination angle in [rad].
chi = 0.0

# Incorporate the inclination angle dependence into a constant.
beta_1 = beta * (
    cfg["k_coefficients"][0] + cfg["k_coefficients"][1] * np.sin(chi) ** 2
)

In [ ]:
# Set this variable to True if you want to also include the X-ray data.
survey_x = True

## Load observed data

### Load the ATNF Pulsar Catalogue for the observed radio surveys.

In [ ]:
surveys_atnf, surveys_meerkat = load_atnf_meerkat_catalog(
    "../../data/observations/atnf_full_nobinary_25-03-2025_with_errors.csv",
    "../../data/observations/meerkat_tpa_posselt_2023.csv"
)

In [ ]:
RA_pk_obs = surveys_atnf["PMPS"]["RA"]
DEC_pk_obs = surveys_atnf["PMPS"]["DEC"]
l_pk_obs = surveys_atnf["PMPS"]["l_gal"]
b_pk_obs = surveys_atnf["PMPS"]["b_gal"]
P_pk_obs = surveys_atnf["PMPS"]["P"]
Pdot_pk_obs = surveys_atnf["PMPS"]["P_dot"]
DM_pk_obs = surveys_atnf["PMPS"]["DM"]
dist_pk_obs = surveys_atnf["PMPS"]["dist"]
S1400_pk_obs = surveys_atnf["PMPS"]["S1400"]
w10_pk_obs = surveys_atnf["PMPS"]["w10"]
pmRA_pk_obs = surveys_atnf["PMPS"]["pm_RA"]
pmDEC_pk_obs = surveys_atnf["PMPS"]["pm_DEC"]

number_pk = len(RA_pk_obs)

In [ ]:
RA_sw_obs = surveys_atnf["SMPS"]["RA"]
DEC_sw_obs = surveys_atnf["SMPS"]["DEC"]
l_sw_obs = surveys_atnf["SMPS"]["l_gal"]
b_sw_obs = surveys_atnf["SMPS"]["b_gal"]
P_sw_obs = surveys_atnf["SMPS"]["P"]
Pdot_sw_obs = surveys_atnf["SMPS"]["P_dot"]
DM_sw_obs = surveys_atnf["SMPS"]["DM"]
dist_sw_obs = surveys_atnf["SMPS"]["dist"]
S1400_sw_obs = surveys_atnf["SMPS"]["S1400"]
w10_sw_obs = surveys_atnf["SMPS"]["w10"]
pmRA_sw_obs = surveys_atnf["SMPS"]["pm_RA"]
pmDEC_sw_obs = surveys_atnf["SMPS"]["pm_DEC"]

number_sw = len(RA_sw_obs)

In [ ]:
RA_htru_obs = surveys_atnf["HTRU_low-mid"]["RA"]
DEC_htru_obs = surveys_atnf["HTRU_low-mid"]["DEC"]
l_htru_obs = surveys_atnf["HTRU_low-mid"]["l_gal"]
b_htru_obs = surveys_atnf["HTRU_low-mid"]["b_gal"]
P_htru_obs = surveys_atnf["HTRU_low-mid"]["P"]
Pdot_htru_obs = surveys_atnf["HTRU_low-mid"]["P_dot"]
DM_htru_obs = surveys_atnf["HTRU_low-mid"]["DM"]
dist_htru_obs = surveys_atnf["HTRU_low-mid"]["dist"]
S1400_htru_obs = surveys_atnf["HTRU_low-mid"]["S1400"]
w10_htru_obs = surveys_atnf["HTRU_low-mid"]["w10"]
pmRA_htru_obs = surveys_atnf["HTRU_low-mid"]["pm_RA"]
pmDEC_htru_obs = surveys_atnf["HTRU_low-mid"]["pm_DEC"]

number_htru = len(RA_htru_obs)

In [ ]:
print(f"Number of pulsars detected by Parks multibeam: {number_pk}")
print(f"Number of pulsars detected by Swinburne: {number_sw}")
print(f"Number of pulsars detected by HTRU: {number_htru}")

### Load the catalog of observed thermally emitting neutron stars.

In [ ]:
surveys_xray = load_xray_catalog(
    "../../data/observations/thermal_NS_05-11-2024.csv",
)

In [ ]:
l_x_obs = surveys_xray["l_gal"]
b_x_obs = surveys_xray["b_gal"]
RA_x_obs = surveys_xray["RA"]
DEC_x_obs = surveys_xray["DEC"]
dist_x_obs = surveys_xray["dist"]
P_x_obs = surveys_xray["P"]
Pdot_x_obs = surveys_xray["P_dot"]
L_x_bol_obs = surveys_xray["L_x_bol"]
S_x_abs_obs = surveys_xray["S_x_abs"]
age_x_obs = surveys_xray["age"]

number_x = len(l_x_obs)

In [ ]:
print(f"Number of pulsars detected in X-rays: {number_x}")

## Load simulated data

### Load the output of a simulated detection with the radio and X-ray surveys.

In [ ]:
path_to_simulation = pathlib.Path("../../data/example_simulation_magrot_det/")

# Load the `.pkl.gz` files containing the survey results to import.
df_PMPS_sim = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
df_PMPS_sim.head()

df_SMPS_sim = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)

df_HTRU_sim = pd.read_pickle(
    pathlib.Path().joinpath(
        path_to_simulation, "survey_HTRU_low_mid_results.pkl.gz"
    ),
    compression="gzip",
)

In [ ]:
# Extracting the parameters.
RA_pk_sim = df_PMPS_sim["ra"]["[deg]"].to_numpy()
DEC_pk_sim = df_PMPS_sim["dec"]["[deg]"].to_numpy()
l_pk_sim = df_PMPS_sim["l"]["[deg]"].to_numpy()
b_pk_sim = df_PMPS_sim["b"]["[deg]"].to_numpy()
pmRA_pk_sim = df_PMPS_sim["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC_pk_sim = df_PMPS_sim["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_pk_sim = df_PMPS_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_pk_sim = df_PMPS_sim["dist"]["[kpc]"].to_numpy()
P_pk_sim = df_PMPS_sim["P"]["[s]"].to_numpy()
Pdot_pk_sim = df_PMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_pk_sim = df_PMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_pk_sim = df_PMPS_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_sw_sim = df_SMPS_sim["ra"]["[deg]"].to_numpy()
DEC_sw_sim = df_SMPS_sim["dec"]["[deg]"].to_numpy()
l_sw_sim = df_SMPS_sim["l"]["[deg]"].to_numpy()
b_sw_sim = df_SMPS_sim["b"]["[deg]"].to_numpy()
pmRA_sw_sim = df_SMPS_sim["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC_sw_sim = df_SMPS_sim["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_sw_sim = df_SMPS_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_sw_sim = df_SMPS_sim["dist"]["[kpc]"].to_numpy()
P_sw_sim = df_SMPS_sim["P"]["[s]"].to_numpy()
Pdot_sw_sim = df_SMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_sw_sim = df_SMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_sw_sim = df_SMPS_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
RA_htru_sim = df_HTRU_sim["ra"]["[deg]"].to_numpy()
DEC_htru_sim = df_HTRU_sim["dec"]["[deg]"].to_numpy()
l_htru_sim = df_HTRU_sim["l"]["[deg]"].to_numpy()
b_htru_sim = df_HTRU_sim["b"]["[deg]"].to_numpy()
pmRA_htru_sim = df_HTRU_sim["pm_ra"]["[mas yr^-1]"].to_numpy()
pmDEC_htru_sim = df_HTRU_sim["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_htru_sim = df_HTRU_sim["DM"]["[pc cm^-3]"].to_numpy()
dist_htru_sim = df_HTRU_sim["dist"]["[kpc]"].to_numpy()
P_htru_sim = df_HTRU_sim["P"]["[s]"].to_numpy()
Pdot_htru_sim = df_HTRU_sim["P_dot"]["[s s^-1]"].to_numpy()
S1400_htru_sim = df_HTRU_sim["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_htru_sim = df_HTRU_sim["w_eff"]["[s]"].to_numpy()

In [ ]:
if survey_x:
    df_x_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_xray_realistic_results.pkl.gz"
        ),
        compression="gzip",
    )
    df_x_sim.columns

    age_x_sim = df_x_sim["age"]["[yr]"].to_numpy()
    RA_x_sim = df_x_sim["ra"]["[deg]"].to_numpy()
    DEC_x_sim = df_x_sim["dec"]["[deg]"].to_numpy()
    l_x_sim = df_x_sim["l"]["[deg]"].to_numpy()
    b_x_sim = df_x_sim["b"]["[deg]"].to_numpy()
    pmRA_x_sim = df_x_sim["pm_ra"]["[mas yr^-1]"].to_numpy()
    pmDEC_x_sim = df_x_sim["pm_dec"]["[mas yr^-1]"].to_numpy()
    NH_x_sim = df_x_sim["N_H"]["[cm^-2]"].to_numpy()
    dist_x_sim = df_x_sim["dist"]["[kpc]"].to_numpy()
    P_x_sim = df_x_sim["P"]["[s]"].to_numpy()
    Pdot_x_sim = df_x_sim["P_dot"]["[s s^-1]"].to_numpy()
    B_x_sim = df_x_sim["B"]["[G]"].to_numpy()
    L_x_sim = df_x_sim["L_x_therm"]["[erg s^-1]"].to_numpy()
    S_x_rcs_abs_sim = df_x_sim["S_x_rcs_abs"]["[erg s^-1 cm^-2]"].to_numpy()
    S_x_bb_abs_sim = df_x_sim["S_x_bb_abs"]["[erg s^-1 cm^-2]"].to_numpy()
    outburst_mask = df_x_sim["outburst"].to_numpy()

In [ ]:
print(
    f"Number of pulsars detected by the simulated Parkes multibeam survey: {len(RA_pk_sim)}"
)
print(
    f"Number of pulsars detected by the simulated Swinburne survey: {len(RA_sw_sim)}"
)
print(
    f"Number of pulsars detected by the simulated HTRU surveys: {len(RA_htru_sim)}"
)
print(
    f"Number of pulsars detected by the simulated HTRU low survey: {len(df_HTRU_sim.loc[(df_HTRU_sim['HTRU_low'] ==1) ])}"
)
print(
    f"Number of pulsars detected by the simulated HTRU mid survey: {len(df_HTRU_sim.loc[(df_HTRU_sim['HTRU_mid'] ==1) ])}"
)
if survey_x:
    print(
        f"Number of pulsars detected by the Simulated X-ray-ray survey: {len(RA_x_sim)}"
    )

### Load the cooling curves from the magneto-thermal simulations.

In [ ]:
base_path = pathlib.Path("../../")

if cfg["magneto-thermal_model"] == "SLy4_dip-tor_heavy":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e12_Btor1e13.csv"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e13_Btor1e14.csv"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e14_Btor1e15.csv"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e15_Btor1e16.csv"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "cool_curve_CC_Bdip5e15_Btor1e16.csv"
    )
elif cfg["magneto-thermal_model"] == "BSk24_dip-tor_heavy":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e12_H.csv"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e13_H.csv"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e14_H.csv"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e15_H.csv"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_5e15_H.csv"
    )
elif cfg["magneto-thermal_model"] == "BSk24_dip-tor_light":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e12_L.csv"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e13_L.csv"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e14_L.csv"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_1e15_L.csv"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "50-50_5e15_L.csv"
    )
elif cfg["magneto-thermal_model"] == "BSk24_multi_heavy":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e12_H.csv"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e13_H.csv"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e14_H.csv"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e15_H.csv"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_5e15_H.csv"
    )
elif cfg["magneto-thermal_model"] == "BSk24_multi_light":
    simB12_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e12_L.csv"
    )
    simB13_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e13_L.csv"
    )
    simB14_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e14_L.csv"
    )
    simB15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_1e15_L.csv"
    )
    simB5e15_path = base_path.joinpath(
        cfg["magneto-thermal_path"], "multi_5e15_L.csv"
    )

In [ ]:
df_B12 = pd.read_csv(
    simB12_path,
    delimiter=",",
    header=[0],
)
df_B12.head()

In [ ]:
df_B13 = pd.read_csv(
    simB13_path,
    delimiter=",",
    header=[0],
)
df_B13.head()

In [ ]:
df_B14 = pd.read_csv(
    simB14_path,
    delimiter=",",
    header=[0],
)
df_B14.head()

In [ ]:
df_B15 = pd.read_csv(
    simB15_path,
    delimiter=",",
    header=[0],
)
df_B15.head()

In [ ]:
df_B5e15 = pd.read_csv(
    simB5e15_path,
    delimiter=",",
    header=[0],
)
df_B5e15.head()

In [ ]:
t12 = df_B12["t[yr]"].to_numpy().astype(float)
t13 = df_B13["t[yr]"].to_numpy().astype(float)
t14 = df_B14["t[yr]"].to_numpy().astype(float)
t15 = df_B15["t[yr]"].to_numpy().astype(float)
t5e15 = df_B5e15["t[yr]"].to_numpy().astype(float)
B12 = df_B12["B[G]"].to_numpy().astype(float)
B13 = df_B13["B[G]"].to_numpy().astype(float)
B14 = df_B14["B[G]"].to_numpy().astype(float)
B15 = df_B15["B[G]"].to_numpy().astype(float)
B5e15 = df_B5e15["B[G]"].to_numpy().astype(float)
L12 = df_B12["L[erg/s]"].to_numpy().astype(float)
L13 = df_B13["L[erg/s]"].to_numpy().astype(float)
L14 = df_B14["L[erg/s]"].to_numpy().astype(float)
L15 = df_B15["L[erg/s]"].to_numpy().astype(float)
L5e15 = df_B5e15["L[erg/s]"].to_numpy().astype(float)

In [ ]:
P_B12 = df_B12["P_fill[s]"].to_numpy().astype(float)
Pdot_B12 = df_B12["Pdot_fill[s/s]"].to_numpy().astype(float)
P_B13 = df_B13["P_fill[s]"].to_numpy().astype(float)
Pdot_B13 = df_B13["Pdot_fill[s/s]"].to_numpy().astype(float)
P_B14 = df_B14["P_fill[s]"].to_numpy().astype(float)
Pdot_B14 = df_B14["Pdot_fill[s/s]"].to_numpy().astype(float)
P_B15 = df_B15["P_fill[s]"].to_numpy().astype(float)
Pdot_B15 = df_B15["Pdot_fill[s/s]"].to_numpy().astype(float)
P_B5e15 = df_B5e15["P_fill[s]"].to_numpy().astype(float)
Pdot_B5e15 = df_B5e15["Pdot_fill[s/s]"].to_numpy().astype(float)

In [ ]:
tau_c_B12 = P_B12 / (2.0 * Pdot_B12) / const.YR_TO_S
tau_c_B13 = P_B13 / (2.0 * Pdot_B13) / const.YR_TO_S
tau_c_B14 = P_B14 / (2.0 * Pdot_B14) / const.YR_TO_S
tau_c_B15 = P_B15 / (2.0 * Pdot_B15) / const.YR_TO_S
tau_c_B5e15 = P_B5e15 / (2.0 * Pdot_B5e15) / const.YR_TO_S

In [ ]:
log_B_range = np.array([12, 13, 14, 15, np.log10(5.0e15)])

## Compare simulations with observations

Comparison of the sky distributions.

In [ ]:
RA_galcen = 266.4
DEC_galcen = -29.0

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    RA_pk_obs,
    DEC_pk_obs,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    RA_sw_obs,
    DEC_sw_obs,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)

ax.plot(
    RA_htru_obs,
    DEC_htru_obs,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed HTRU",
)
if survey_x:
    ax.plot(
        RA_x_obs,
        DEC_x_obs,
        linestyle="None",
        marker="X",
        color="tab:purple",
        markersize=6,
        rasterized=True,
        label=r"Observed X-ray",
    )

ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    RA_pk_sim,
    DEC_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    RA_sw_sim,
    DEC_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)

ax.plot(
    RA_htru_sim,
    DEC_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    rasterized=True,
    label=r"Simulated HTRU",
)
if survey_x:
    ax.plot(
        RA_x_sim,
        DEC_x_sim,
        linestyle="None",
        marker="X",
        color="tab:purple",
        markersize=6,
        rasterized=True,
        label=r"Simulated X-ray",
    )

ax.plot(RA_galcen, DEC_galcen, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(0.0, 360.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("RA [deg]")
ax.set_ylabel("DEC [deg]")
ax.legend(frameon=True, loc=0)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_pk_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    RA_pk_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)


ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_sw_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    RA_sw_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=True, loc=2)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

RA_edges = np.linspace(0.0, 360.0, 31)

ax.hist(
    RA_htru_obs,
    bins=RA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    RA_htru_sim,
    bins=RA_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"RA [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(0.0, 360.0)
ax.legend(frameon=True, loc=2)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    RA_edges = np.linspace(0.0, 360.0, 31)

    ax.hist(
        RA_x_obs,
        bins=RA_edges,
        histtype="stepfilled",
        edgecolor="darkgray",
        facecolor="darkgray",
        lw=4,
        alpha=1,
        label=r"Observed X-ray",
    )
    ax.hist(
        RA_x_sim,
        bins=RA_edges,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        alpha=1,
        label=r"Simulated X-ray",
    )
    ax.set_xlabel(r"RA [deg]")
    ax.set_ylabel(r"Number of NSs")
    ax.set_xlim(0.0, 360.0)
    ax.legend(frameon=True, loc=2)

    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_pk_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    DEC_pk_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_sw_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    DEC_sw_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

DEC_edges = np.linspace(-90.0, 90.0, 31)

ax.hist(
    DEC_htru_obs,
    bins=DEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    DEC_htru_sim,
    bins=DEC_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"DEC [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    DEC_edges = np.linspace(-90.0, 90.0, 31)

    ax.hist(
        DEC_x_obs,
        bins=DEC_edges,
        histtype="stepfilled",
        edgecolor="darkgray",
        facecolor="darkgray",
        lw=4,
        alpha=1,
        label=r"Observed X-ray",
    )
    ax.hist(
        DEC_x_sim,
        bins=DEC_edges,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        alpha=1,
        label=r"Simulated X-ray",
    )
    ax.set_xlabel(r"DEC [deg]")
    ax.set_ylabel(r"Number of NSs")
    ax.set_xlim(-90.0, 90.0)
    ax.legend(frameon=True, loc=2)

    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_obs,
    b_pk_obs,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    l_sw_obs,
    b_sw_obs,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    l_htru_obs,
    b_htru_obs,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Observed HTRU",
)

if survey_x:
    ax.plot(
        l_x_obs,
        b_x_obs,
        linestyle="None",
        marker="X",
        color="tab:purple",
        markersize=6,
        alpha=1,
        rasterized=True,
        label=r"Observed X-ray",
    )

ax.plot(0.0, 0.0, marker="*", color="tab:blue", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_sim,
    b_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    l_sw_sim,
    b_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    l_htru_sim,
    b_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated HTRU",
)

if survey_x:
    ax.plot(
        l_x_sim,
        b_x_sim,
        linestyle="None",
        marker="X",
        color="tab:purple",
        markersize=6,
        rasterized=True,
        label=r"Simulated X-ray",
    )

ax.plot(0.0, 0.0, marker="*", color="tab:blue", markersize=20)
ax.set_xlim(-180.0, 180.0)
ax.set_ylim(-90.0, 90.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
l_edges = np.linspace(-180.0, 180.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_pk_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    l_pk_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_sw_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    l_sw_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=True, loc=2)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    l_htru_obs,
    bins=l_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    l_htru_sim,
    bins=l_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"l [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-180.0, 180.0)
ax.legend(frameon=True, loc=2)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        l_x_obs,
        bins=l_edges,
        histtype="stepfilled",
        edgecolor="darkgray",
        facecolor="darkgray",
        lw=4,
        alpha=1,
        label=r"Observed X-ray",
    )
    ax.hist(
        l_x_sim,
        bins=l_edges,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        alpha=1,
        label=r"Simulated X-ray",
    )
    ax.set_xlabel(r"l [deg]")
    ax.set_ylabel(r"Number of NSs")
    ax.set_xlim(-180.0, 180.0)
    ax.legend(frameon=True, loc=2)

    plt.show()

In [ ]:
b_edges = np.linspace(-40.0, 40.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_pk_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    b_pk_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_sw_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    b_sw_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=True, loc=2)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    b_htru_obs,
    bins=b_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    b_htru_sim,
    bins=b_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"b [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_xlim(-40.0, 40.0)
ax.legend(frameon=True, loc=2)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        b_x_obs,
        bins=b_edges,
        histtype="stepfilled",
        edgecolor="darkgray",
        facecolor="darkgray",
        lw=4,
        alpha=1,
        label=r"Observed X-ray",
    )
    ax.hist(
        b_x_sim,
        bins=b_edges,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        alpha=1,
        label=r"Simulated X-ray",
    )
    ax.set_xlabel(r"b [deg]")
    ax.set_ylabel(r"Number of NSs")
    ax.set_xlim(-40.0, 40.0)
    ax.legend(frameon=True, loc=2)

    plt.show()

Compare proper motion distributions.

In [ ]:
pmRA_edges = np.linspace(-90.0, 90.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_pk_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    pmRA_pk_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_sw_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    pmRA_sw_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmRA_htru_obs,
    bins=pmRA_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    pmRA_htru_sim,
    bins=pmRA_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"$\mu_{\rm RA}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
pmDEC_edges = np.linspace(-90.0, 90.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_pk_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed PMPS",
)
ax.hist(
    pmDEC_pk_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Simulated PMPS",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_sw_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed SMPS",
)
ax.hist(
    pmDEC_sw_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated SMPS",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    pmDEC_htru_obs,
    bins=pmDEC_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Observed HTRU",
)
ax.hist(
    pmDEC_htru_sim,
    bins=pmDEC_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Simulated HTRU",
)
ax.set_xlabel(r"$\mu_{\rm DEC}$ [deg]")
ax.set_ylabel(r"Number of NSs")
ax.set_yscale("log")
ax.set_xlim(-90.0, 90.0)
ax.legend(frameon=True, loc=0)

plt.show()

Compare DM distributions.

In [ ]:
dm_edges = np.linspace(0, 2500, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_pk_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS",
    rasterized=True,
)
ax.hist(
    DM_pk_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Simulated PMPS",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
# ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_sw_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS",
    rasterized=True,
)
ax.hist(
    DM_sw_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.0,
    label=r"Simulated SMPS",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
# ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    DM_htru_obs,
    bins=dm_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed HTRU",
    rasterized=True,
)
ax.hist(
    DM_htru_sim,
    bins=dm_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1.0,
    label=r"Simulated HTRU",
    rasterized=True,
)
ax.set_xlabel(r"DM [pc cm$^{-3}$]")
ax.set_ylabel("Number of NSs")
# ax.set_yscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

Compare distance distributions.

In [ ]:
d_edges = np.linspace(0, 30, 36)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_pk_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed PMPS",
    rasterized=True,
)
ax.hist(
    dist_pk_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1.0,
    label=r"Simulated PMPS",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_sw_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed SMPS",
    rasterized=True,
)
ax.hist(
    dist_sw_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1.0,
    label=r"Simulated SMPS",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=True, loc=0)

plt.show()


fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist_htru_obs,
    bins=d_edges,
    histtype="stepfilled",
    edgecolor="darkgray",
    facecolor="darkgray",
    lw=4,
    alpha=1.0,
    label=r"Observed HTRU",
    rasterized=True,
)
ax.hist(
    dist_htru_sim,
    bins=d_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1.0,
    label=r"Simulated HTRU",
    rasterized=True,
)
ax.set_xlabel(r"$d_{\odot}$ [kpc]")
ax.set_ylabel("Number of NSs")
ax.legend(frameon=True, loc=0)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        dist_x_obs,
        bins=d_edges,
        histtype="stepfilled",
        edgecolor="darkgray",
        facecolor="darkgray",
        lw=4,
        alpha=1.0,
        label=r"Observed X-ray",
        rasterized=True,
    )
    ax.hist(
        dist_x_sim,
        bins=d_edges,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        alpha=1.0,
        label=r"Simulated X-ray",
        rasterized=True,
    )
    ax.set_xlabel(r"$d_{\odot}$ [kpc]")
    ax.set_ylabel("Number of NSs")
    ax.legend(frameon=True, loc=0)

    plt.show()

Comparing the spin-period and spin-period-derivative distributions.

In [ ]:
P_bins = np.logspace(-2.0, 2.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_pk_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    P_pk_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_sw_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    P_sw_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P_htru_obs,
    bins=P_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    P_htru_sim,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        P_x_obs,
        bins=P_bins,
        histtype="stepfilled",
        color="darkgray",
        lw=4,
        label="Observed X-ray",
    )
    ax.hist(
        P_x_sim,
        bins=P_bins,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        label="Simulated X-ray",
    )

    plt.xlabel(r"$P$ [s]")
    plt.ylabel(r"Number of NSs")
    plt.xscale("log")
    ax.legend(frameon=True, loc=0)

    plt.show()

In [ ]:
Pdot_bins = np.logspace(-20.0, -8.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_pk_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    Pdot_pk_sim,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_sw_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    Pdot_sw_sim,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Pdot_htru_obs,
    bins=Pdot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    Pdot_htru_sim,
    bins=Pdot_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)

plt.xlabel(r"$\dot{P}$ [s]")
plt.ylabel(r"Number of NSs")
plt.xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        Pdot_x_obs,
        bins=Pdot_bins,
        histtype="stepfilled",
        color="darkgray",
        lw=4,
        label="Observed X-ray",
    )
    ax.hist(
        Pdot_x_sim,
        bins=Pdot_bins,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        label="Simulated X-ray",
    )

    plt.xlabel(r"$\dot{P}$ [s]")
    plt.ylabel(r"Number of NSs")
    plt.xscale("log")
    ax.legend(frameon=True, loc=0)

    plt.show()

Comparing PPdot diagrams.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_obs,
    Pdot_pk_obs,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    P_sw_obs,
    Pdot_sw_obs,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    P_htru_obs,
    Pdot_htru_obs,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Observed HTRU",
)

if survey_x:
    ax.plot(
        P_x_obs,
        Pdot_x_obs,
        linestyle="None",
        marker="X",
        color="tab:purple",
        markersize=15,
        alpha=1.0,
        rasterized=True,
        label=r"Observed X-ray",
    )


ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-3, 100.0)
ax.set_ylim(1.0e-21, 1.0e-9)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=2)

plt.grid()

fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_sim,
    Pdot_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    P_sw_sim,
    Pdot_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    P_htru_sim,
    Pdot_htru_sim,
    linestyle="None",
    marker="o",
    fillstyle="none",
    color="tab:green",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated HTRU",
)

if survey_x:
    ax.plot(
        P_x_sim,
        Pdot_x_sim,
        linestyle="None",
        marker="X",
        color="tab:purple",
        markersize=15,
        alpha=1.0,
        rasterized=True,
        label=r"Simulated X-ray",
    )
    ax.plot(
        P_x_sim[outburst_mask],
        Pdot_x_sim[outburst_mask],
        linestyle="None",
        marker="X",
        color="black",
        markersize=10,
        alpha=1.0,
        rasterized=True,
        label=r"Simulated X-ray (outburst)",
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-3, 100.0)
ax.set_ylim(1.0e-21, 1.0e-9)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=0)

plt.grid()

Comparing the spin-down power distributions.

In [ ]:
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2

# Computing the spin-down power.
Erot_dot_pk_sim = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_pk_sim / (P_pk_sim**3)
)
Erot_dot_sw_sim = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_sw_sim / (P_sw_sim**3)
)
Erot_dot_htru_sim = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_htru_sim / (P_htru_sim**3)
)
if survey_x:
    Erot_dot_x_sim = (
        NS_inertia * (2.0 * np.pi) ** 2 * Pdot_x_sim / (P_x_sim**3)
    )

Erot_dot_pk_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_pk_obs / (P_pk_obs**3)
)
Erot_dot_sw_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_sw_obs / (P_sw_obs**3)
)
Erot_dot_htru_obs = (
    NS_inertia * (2.0 * np.pi) ** 2 * Pdot_htru_obs / (P_htru_obs**3)
)
if survey_x:
    Erot_dot_x_obs = (
        NS_inertia * (2.0 * np.pi) ** 2 * Pdot_x_obs / (P_x_obs**3)
    )

In [ ]:
Erot_dot_bins = np.logspace(27.0, 40.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_pk_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    Erot_dot_pk_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_sw_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    Erot_dot_sw_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot_htru_obs,
    bins=Erot_dot_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    Erot_dot_htru_sim,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        Erot_dot_x_obs,
        bins=Erot_dot_bins,
        histtype="stepfilled",
        color="darkgray",
        lw=4,
        label="Observed X-ray",
    )
    ax.hist(
        Erot_dot_x_sim,
        bins=Erot_dot_bins,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        label="Simulated X-ray",
    )
    plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
    plt.ylabel(r"Number of NSs")
    ax.set_xscale("log")
    ax.legend(frameon=True, loc=0)

    plt.show()

Comparing the characteristic age distributions.

In [ ]:
age_char_pk_obs = P_pk_obs / (2 * Pdot_pk_obs) / const.YR_TO_S
age_char_sw_obs = P_sw_obs / (2 * Pdot_sw_obs) / const.YR_TO_S
age_char_htru_obs = P_htru_obs / (2 * Pdot_htru_obs) / const.YR_TO_S
age_char_x_obs = P_x_obs / (2 * Pdot_x_obs) / const.YR_TO_S

age_char_pk_sim = P_pk_sim / (2 * Pdot_pk_sim) / const.YR_TO_S
age_char_sw_sim = P_sw_sim / (2 * Pdot_sw_sim) / const.YR_TO_S
age_char_htru_sim = P_htru_sim / (2 * Pdot_htru_sim) / const.YR_TO_S

if survey_x:
    age_char_x_sim = P_x_sim / (2 * Pdot_x_sim) / const.YR_TO_S

In [ ]:
age_char_bins = np.logspace(2, 11.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    age_char_pk_obs,
    bins=age_char_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    age_char_pk_sim,
    bins=age_char_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$\tau_{\rm c}$ [yr]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    age_char_sw_obs,
    bins=age_char_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    age_char_sw_sim,
    bins=age_char_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$\tau_{\rm c}$ [yr]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    age_char_htru_obs,
    bins=age_char_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    age_char_htru_sim,
    bins=age_char_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$\tau_{\rm c}$ [yr]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        age_char_x_obs,
        bins=age_char_bins,
        histtype="stepfilled",
        color="darkgray",
        lw=4,
        label="Observed X-ray",
    )
    ax.hist(
        age_char_x_sim,
        bins=age_char_bins,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        label="Simulated X-ray",
    )
    plt.xlabel(r"$\tau_{\rm c}$ [yr]")
    plt.ylabel(r"Number of NSs")
    ax.set_xscale("log")
    ax.legend(frameon=True, loc=2)

    plt.show()

Comparing the magnetic field distributions assuming inclination angle $\chi = 0$ deg.

In [ ]:
B_pk_obs = np.sqrt(1.0 / beta_1 * P_pk_obs * Pdot_pk_obs)
B_sw_obs = np.sqrt(1.0 / beta_1 * P_sw_obs * Pdot_sw_obs)
B_htru_obs = np.sqrt(1.0 / beta_1 * P_htru_obs * Pdot_htru_obs)
B_x_obs = np.sqrt(1.0 / beta_1 * P_x_obs * Pdot_x_obs)

B_pk_sim = np.sqrt(1.0 / beta_1 * P_pk_sim * Pdot_pk_sim)
B_sw_sim = np.sqrt(1.0 / beta_1 * P_sw_sim * Pdot_sw_sim)
B_htru_sim = np.sqrt(1.0 / beta_1 * P_htru_sim * Pdot_htru_sim)
if survey_x:
    B_x_sim = np.sqrt(1.0 / beta_1 * P_x_sim * Pdot_x_sim)

In [ ]:
B_bins = np.logspace(9.0, 17.0, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    B_pk_obs,
    bins=B_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    B_pk_sim,
    bins=B_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$B$ [G]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    B_sw_obs,
    bins=B_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    B_sw_sim,
    bins=B_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$B$ [G]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    B_htru_obs,
    bins=B_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    B_htru_sim,
    bins=B_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$B$ [G]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=2)

plt.show()

if survey_x:
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        B_x_obs,
        bins=B_bins,
        histtype="stepfilled",
        color="darkgray",
        lw=4,
        label="Observed X-ray",
    )
    ax.hist(
        B_x_sim,
        bins=B_bins,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        label="Simulated X-ray",
    )
    plt.xlabel(r"$B$ [G]")
    plt.ylabel(r"Number of NSs")
    ax.set_xscale("log")
    ax.legend(frameon=True, loc=2)

    plt.show()

Comparing the radio-flux distributions.

In [ ]:
S_radio_bins = np.logspace(-5, 1, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_pk_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed PMPS",
)
ax.hist(
    S1400_pk_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_sw_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed SMPS",
)
ax.hist(
    S1400_sw_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_htru_obs / 1000.0,
    bins=S_radio_bins,
    histtype="stepfilled",
    color="darkgray",
    lw=4,
    label="Observed HTRU",
)
ax.hist(
    S1400_htru_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

In [ ]:
L_pseudo_radio_pk_obs = (
    S1400_pk_obs
    * const.MILLIJY_TO_ERG
    * (dist_pk_obs * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_sw_obs = (
    S1400_sw_obs
    * const.MILLIJY_TO_ERG
    * (dist_sw_obs * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_htru_obs = (
    S1400_htru_obs
    * const.MILLIJY_TO_ERG
    * (dist_htru_obs * const.KPC_TO_CM) ** 2
    * 1.374e9
)

L_pseudo_radio_pk_sim = (
    S1400_pk_sim
    * 1000
    * const.MILLIJY_TO_ERG
    * (dist_pk_sim * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_sw_sim = (
    S1400_sw_sim
    * 1000
    * const.MILLIJY_TO_ERG
    * (dist_sw_sim * const.KPC_TO_CM) ** 2
    * 1.374e9
)
L_pseudo_radio_htru_sim = (
    S1400_htru_sim
    * 1000
    * const.MILLIJY_TO_ERG
    * (dist_htru_sim * const.KPC_TO_CM) ** 2
    * 1.374e9
)

eff_radio_pk_obs = L_pseudo_radio_pk_obs / Erot_dot_pk_obs
eff_radio_sw_obs = L_pseudo_radio_sw_obs / Erot_dot_sw_obs
eff_radio_htru_obs = L_pseudo_radio_htru_obs / Erot_dot_htru_obs

eff_radio_pk_sim = L_pseudo_radio_pk_sim / Erot_dot_pk_sim
eff_radio_sw_sim = L_pseudo_radio_sw_sim / Erot_dot_sw_sim
eff_radio_htru_sim = L_pseudo_radio_htru_sim / Erot_dot_htru_sim

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")

ax.loglog(
    Erot_dot_pk_obs,
    eff_radio_pk_obs,
    "o",
    color="darkgray",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed PMPS",
)
ax.loglog(
    Erot_dot_pk_sim,
    eff_radio_pk_sim,
    "o",
    color="tab:red",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated PMPS",
)

ax.axhline(y=1, color="black", linestyle="-", linewidth=2)
ax.axhline(y=0.1, color="black", linestyle="--", linewidth=2)

plt.legend(frameon=True, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")

ax.loglog(
    Erot_dot_sw_obs,
    eff_radio_sw_obs,
    "o",
    color="darkgray",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed SMPS",
)
ax.loglog(
    Erot_dot_sw_sim,
    eff_radio_sw_sim,
    "o",
    color="tab:blue",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated SMPS",
)
ax.axhline(y=1, color="black", linestyle="-", linewidth=2)
ax.axhline(y=0.1, color="black", linestyle="--", linewidth=2)

plt.legend(frameon=True, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")

ax.loglog(
    Erot_dot_htru_obs,
    eff_radio_htru_obs,
    "o",
    color="darkgray",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed HTRU",
)
ax.loglog(
    Erot_dot_htru_sim,
    eff_radio_htru_sim,
    "o",
    color="tab:green",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated HTRU",
)

ax.axhline(y=1, color="black", linestyle="-", linewidth=2)
ax.axhline(y=0.1, color="black", linestyle="--", linewidth=2)

plt.legend(frameon=True, loc=0, fontsize=20)

plt.show()

In [ ]:
if survey_x:
    S_x_bins = np.logspace(-15.0, -9, 21)

    fig, ax = plt.subplots(figsize=(15, 8))

    ax.hist(
        S_x_abs_obs,
        bins=S_x_bins,
        histtype="stepfilled",
        color="darkgray",
        lw=4,
        label="Observed X-ray",
    )
    ax.hist(
        S_x_rcs_abs_sim,
        bins=S_x_bins,
        histtype="step",
        edgecolor="tab:purple",
        lw=4,
        label="Simulated X-ray",
    )
    plt.xlabel(r"$S_{\rm X, abs}$ [erg s$^{-1}$ cm$^{-2}$]")
    plt.ylabel(r"Number of NSs")
    ax.set_xscale("log")
    ax.legend(frameon=True, loc=0)

    plt.show()

In [ ]:
if survey_x:
    # Sort the flux densities in ascending order
    sorted_S_x_sim = np.sort(S_x_rcs_abs_sim)
    sorted_S_x_obs = np.sort(S_x_abs_obs)

    # Calculate the cumulative number of sources
    cumulative_number_sim = np.arange(len(sorted_S_x_sim), 0, -1)
    cumulative_number_obs = np.arange(len(sorted_S_x_obs), 0, -1)

In [ ]:
if survey_x:
    # Create the log N - log S step plot
    fig, ax = plt.subplots(figsize=(15, 8))

    ax.step(
        sorted_S_x_sim,
        cumulative_number_sim,
        where="mid",
        lw=4,
        color="tab:purple",
    )
    ax.plot(
        sorted_S_x_obs,
        (sorted_S_x_obs / 5.0e-11) ** (-3.0 / 2.0),
        lw=4,
        ls=":",
        color="tab:gray",
    )
    ax.step(
        sorted_S_x_obs, cumulative_number_obs, where="mid", lw=4, color="black"
    )

    # Set logarithmic scale
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_ylim(0.5, 1.0e2)

    # Adding titles and labels
    plt.xlabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
    plt.ylabel(r"$N(>S_{X, \rm{abs}})$")

    # Add grid lines
    plt.grid(True, which="both", ls="--")

    # Show the plot
    plt.show()

In [ ]:
if survey_x:
    vmin, vmax = np.min(log_B_range), np.max(log_B_range)

    # Create a colormap and normalize it.
    cmap = plt.cm.viridis
    norm = Normalize(vmin=vmin, vmax=vmax)

    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    fig, ax = plt.subplots(figsize=(12, 10))

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(20.0, 1.0e8)
    # ax.set_ylim(Pdot_min,Pdot_max)
    ax.set_xlabel(r"$\tau_{\rm c}$ [yr]")
    ax.set_ylabel(r"$L_{X}$ [erg s$^{-1}$]")

    ax.plot(
        tau_c_B12,
        L12,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[0])),
        rasterized=True,
    )
    ax.plot(
        tau_c_B13,
        L13,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[1])),
        rasterized=True,
    )
    ax.plot(
        tau_c_B14,
        L14,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[2])),
        rasterized=True,
    )
    ax.plot(
        tau_c_B15,
        L15,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[3])),
        rasterized=True,
    )
    ax.plot(
        tau_c_B5e15,
        L5e15,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[4])),
        rasterized=True,
    )

    ax.plot(
        age_char_x_obs,
        L_x_bol_obs,
        marker="X",
        color="darkgray",
        linestyle="None",
        markersize=15,
        alpha=1,
        rasterized=True,
        label="Observed X-ray",
    )
    ax.plot(
        age_char_x_sim,
        L_x_sim,
        marker="o",
        color="tab:purple",
        linestyle="None",
        markersize=6,
        alpha=1,
        rasterized=True,
        label="Simulated X-ray",
    )

    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

    plt.legend(frameon=True, loc=0, fontsize=20)

    plt.grid()

NOTE: For calculating the characteristic age for the cooling curves we considered the value of $P$ and $\dot{P}$ computed by solving equations (10) and (11) in [Graber et al. 2024](https://ui.adsabs.harvard.edu/abs/2024ApJ...968...16G/abstract) and assuming an initial spin period of 0.001 s, an initial inclination angle of 60 deg and the magnetic field evolution associated with the cooling curves.

In [ ]:
if survey_x:
    fig, ax = plt.subplots(figsize=(12, 10))

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1.0, 1.0e8)
    # ax.set_ylim(Pdot_min,Pdot_max)
    ax.set_xlabel(r"Real age [yr]")
    ax.set_ylabel(r"$L_{X}$ [erg s$^{-1}$]")

    ax.plot(
        t12,
        L12,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[0])),
        rasterized=True,
    )
    ax.plot(
        t13,
        L13,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[1])),
        rasterized=True,
    )
    ax.plot(
        t14,
        L14,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[2])),
        rasterized=True,
    )
    ax.plot(
        t15,
        L15,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[3])),
        rasterized=True,
    )
    ax.plot(
        t5e15,
        L5e15,
        linestyle="-",
        linewidth=4,
        color=cmap(norm(log_B_range[4])),
        rasterized=True,
    )

    ax.plot(
        age_x_obs * 1000,
        L_x_bol_obs,
        marker="X",
        color="darkgray",
        linestyle="None",
        markersize=15,
        alpha=1,
        rasterized=True,
        label="Observed X-ray",
    )
    ax.plot(
        age_x_sim,
        L_x_sim,
        marker="o",
        color="tab:purple",
        linestyle="None",
        markersize=6,
        alpha=1,
        rasterized=True,
        label="Simulated X-ray",
    )

    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

    plt.legend(frameon=True, loc=0, fontsize=20)

    plt.grid()

In [ ]:
if survey_x:
    fig, ax = plt.subplots(figsize=(12, 10))

    ax.set_xlabel(r"$B$ [G]")
    ax.set_ylabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.plot(
        B_x_obs,
        S_x_abs_obs,
        marker="X",
        color="darkgray",
        linestyle="None",
        markersize=15,
        alpha=1,
        rasterized=True,
        label="Observed X-ray",
    )
    ax.plot(
        B_x_sim,
        S_x_rcs_abs_sim,
        marker="o",
        color="tab:purple",
        linestyle="None",
        markersize=6,
        alpha=1,
        rasterized=True,
        label="Simulated X-ray",
    )

    plt.legend(frameon=True, loc=0, fontsize=20)

    plt.grid()

In [ ]:
if survey_x:
    fig, ax = plt.subplots(figsize=(12, 10))

    ax.set_xlabel(r"$P$ [s]")
    ax.set_ylabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.plot(
        P_x_obs,
        S_x_abs_obs,
        marker="X",
        color="darkgray",
        linestyle="None",
        markersize=15,
        alpha=1,
        rasterized=True,
        label="Observed X-ray",
    )
    ax.plot(
        P_x_sim,
        S_x_rcs_abs_sim,
        marker="o",
        color="tab:purple",
        linestyle="None",
        markersize=6,
        alpha=1,
        rasterized=True,
        label="Simulated X-ray",
    )

    plt.legend(frameon=True, loc=0, fontsize=20)

    plt.grid()

In [ ]:
if survey_x:
    fig, ax = plt.subplots(figsize=(12, 10))

    ax.set_xlabel(r"$\dot{P}$ [s s$^{-1}$]")
    ax.set_ylabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.plot(
        Pdot_x_obs,
        S_x_abs_obs,
        marker="X",
        color="darkgray",
        linestyle="None",
        markersize=15,
        alpha=1,
        rasterized=True,
        label="Observed X-ray",
    )
    ax.plot(
        Pdot_x_sim,
        S_x_rcs_abs_sim,
        marker="o",
        color="tab:purple",
        linestyle="None",
        markersize=6,
        alpha=1,
        rasterized=True,
        label="Simulated X-ray",
    )

    plt.legend(frameon=True, loc=0, fontsize=20)

    plt.grid()

In [ ]:
if survey_x:
    fig, ax = plt.subplots(figsize=(12, 10))

    ax.set_xlabel(r"$d$ [kpc]")
    ax.set_ylabel(r"$B$ [G]")
    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.plot(
        dist_x_obs,
        B_x_obs,
        marker="X",
        color="darkgray",
        linestyle="None",
        markersize=15,
        alpha=1,
        rasterized=True,
        label="Observed X-ray",
    )
    ax.plot(
        dist_x_sim,
        B_x_sim,
        marker="o",
        color="tab:purple",
        linestyle="None",
        markersize=6,
        alpha=1,
        rasterized=True,
        label="Simulated X-ray",
    )

    plt.legend(frameon=True, loc=0, fontsize=20)

    plt.grid()

Comparing the pulse-width distributions.

In [ ]:
# Converting pulse width into [deg].
w_pk_sim_deg = w_eff_pk_sim / P_pk_sim * 360.0
w_sw_sim_deg = w_eff_sw_sim / P_sw_sim * 360.0
w_htru_sim_deg = w_eff_htru_sim / P_htru_sim * 360.0

w10_pk_obs_deg = w10_pk_obs / 1000 / P_pk_obs * 360.0
w10_sw_obs_deg = w10_sw_obs / 1000 / P_sw_obs * 360.0
w10_htru_obs_deg = w10_htru_obs / 1000 / P_htru_obs * 360.0

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(
    P_pk_obs,
    w10_pk_obs_deg,
    "o",
    color="tab:red",
    ms=6,
    rasterized=True,
    label="Observed PMPS",
)
ax.loglog(
    P_sw_obs,
    w10_sw_obs_deg,
    "o",
    color="tab:blue",
    ms=6,
    rasterized=True,
    label="Observed SMPS",
)

ax.loglog(
    P_htru_obs,
    w10_htru_obs_deg,
    "o",
    fillstyle="none",
    color="tab:green",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Observed HTRU",
)
plt.xlim(3.0e-2, 20)
plt.ylim(1, 400)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")
ax.legend(frameon=True, loc=3)


fig, ax = plt.subplots(figsize=(15, 8))

ax.loglog(
    P_pk_sim,
    w_pk_sim_deg,
    "o",
    color="tab:red",
    ms=6,
    rasterized=True,
    label="Simulated PMPS",
)
ax.loglog(
    P_sw_sim,
    w_sw_sim_deg,
    "o",
    color="tab:blue",
    ms=6,
    rasterized=True,
    label="Simulated SMPS",
)
ax.loglog(
    P_htru_sim,
    w_htru_sim_deg,
    "o",
    fillstyle="none",
    color="tab:green",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Simulated HTRU",
)

plt.xlim(3.0e-2, 20)
plt.ylim(1, 400)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")
ax.legend(frameon=True, loc=3)

plt.show()